# HCP-HCO Affiliation Mapping

**Goal:** Keep **all raw HCP rows** and add helper flags/fields to quickly tell whether an NPI has at least one valid, verified address and what the KOMODO/territory context is.

**What it builds**

Creates/overwrites a **temporary view** hcp_info from these sources:

- target_hcp_hco_mapping a → NPIs + basic HCP identity seed
- vod_hcp b → Veeva HCP vid_\_v (joins on NPI)
- vod_address c → HCP addresses (HCP entity, **VALID** records, status in **A/DS**, **verification not in NS/U**)
- zip_to_territory_mapping d → Territory/region for VOD address ZIP
- kom_providers e → KOM provider (INDIVIDUAL) address + ZIP
- zip_to_territory_mapping f → Territory/region for KOM ZIP

**Key columns produced**

From **VOD/HCP**:

- hcp_npi, hcp_vid, hcp_name
- Address fields: hcp_address_modified_date, hcp_address_line_1, hcp_zip, hcp_city, hcp_state
- Address metadata: hcp_address_type, hcp_address_verification_status, hcp_address_vid

From **zip-to-territory** (VOD side):

- hcp_terr, hcp_region

From **KOMODO**:

- kom_hcp_address_line_1, kom_hcp_zip, kom_hcp_city, kom_hcp_state
- kom_hcp_terr, kom_hcp_region

**Row-level address quality flag**

- hcp_address_flag = **1** if c.record_state_\_v = 'VALID' **and** c.address_status_\_v IN ('A','DS')  
    (only rows from c that already pass these filters appear; this is restated for clarity)

**NPI-level helper flags (window over PARTITION BY hcp_npi)**

- hcp_address_flag_1 = **MAX(row hcp_address_flag)** → **1 if the NPI has at least one good address**, else 0
- hcp_address_flag_0 = **1 if the NPI has no good addresses at all** (inverse convenience flag)

**Logic flow (at a glance)**

- Start from target NPIs (a).
- Get Veeva HCP ID (b.vid_\_v) via NPI match.
- Pull HCP addresses (c) for that VID with **entity_type='HCP'**, **VALID** records, **status A/DS**, and **verification not NS/U**.
- Map VOD ZIP → territory/region (d).
- Pull KOM provider record for the same NPI (e, INDIVIDUAL only) and map its ZIP → territory/region (f).
- Add row-level hcp_address_flag, then compute per-NPI window flags hcp_address_flag_1 and hcp_address_flag_0.
- Keep **every row**; order by hcp_npi, newest hcp_address_modified_date first.

**Why the two flags?**

- Use hcp_address_flag_1 = 1 to **retain** all rows for NPIs that have **any** acceptable address (while keeping raw duplicates/variants).
- Use hcp_address_flag_0 = 1 to quickly **identify NPIs with zero acceptable addresses** across all their rows.

In [0]:
%sql
-- Rebuild the view with a group-level flag
-- (keeps ALL raw rows; just adds helper columns)
CREATE OR REPLACE TEMPORARY VIEW hcp_info AS
WITH base AS (
  SELECT
      a.hcp_npi,
      b.vid__v AS hcp_vid,
      CONCAT(a.hcp_first_name, ' ', a.hcp_last_name) AS hcp_name,
      CAST(c.modified_date__v AS DATE) AS hcp_address_modified_date,
      c.address_line_1__v AS hcp_address_line_1,
      c.postal_code_cda__v AS hcp_zip,
      c.city_cda__v AS hcp_city,
      c.state_cda__v AS hcp_state,
      d.territory_name AS hcp_terr,
      d.region_name  AS hcp_region,
      c.address_type__v AS hcp_address_type,
      c.address_verification_status__v AS hcp_address_verification_status,
      c.vid__v AS hcp_address_vid,
      e.PROVIDER_ADDRESS AS kom_hcp_address_line_1,
      e.PROVIDER_ZIP     AS kom_hcp_zip,
      e.PROVIDER_CITY    AS kom_hcp_city,
      e.PROVIDER_STATE   AS kom_hcp_state,
      f.territory_name   AS kom_hcp_terr,
      f.region_name      AS kom_hcp_region,
      CASE
        WHEN c.record_state__v = 'VALID' AND c.address_status__v IN ('A','DS') THEN 1
        ELSE 0
      END AS hcp_address_flag
  FROM com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping a
  LEFT JOIN com_edp_prd.com_raw.vod_hcp b
    ON a.hcp_npi = TRY_CAST(b.npi_num__v AS BIGINT)
  LEFT JOIN com_edp_prd.com_raw.vod_address c
    ON b.vid__v = c.entity_vid__v
   AND c.entity_type__v = 'HCP'
   AND c.record_state__v = 'VALID'
   AND c.address_status__v IN ('A','DS')
   AND c.address_verification_status__v NOT IN ('NS', 'U') -- address_verification_status = Not Specified / Unverified
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping d
    ON c.postal_code_cda__v = d.zipcode
  LEFT JOIN com_edp_prd.com_raw.kom_providers e
    ON a.hcp_npi = e.NPI
   AND e.PROVIDER_TYPE = 'INDIVIDUAL'
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping f
    ON e.PROVIDER_ZIP = f.zipcode
)
SELECT
  b.*,
  -- Any row with flag=1 for this NPI?
  MAX(COALESCE(b.hcp_address_flag,0)) OVER (PARTITION BY b.hcp_npi) AS hcp_address_flag_1,
  -- Convenience inverse: 1 means ALL rows are zero for this NPI
  CASE
    WHEN MAX(COALESCE(b.hcp_address_flag,0)) OVER (PARTITION BY b.hcp_npi) = 0 THEN 1
    ELSE 0
  END AS hcp_address_flag_0
FROM base b
ORDER BY b.hcp_npi ASC, b.hcp_address_modified_date DESC;


## hcp_info_dedup - Purpose

Selects **one best address per HCP (NPI)** from the raw HCP view (hcp_info).  
Prioritizes **KOM-matching ZIP** and then **latest address** if there's a tie.  
Ensures you have a **single clean address row per HCP** for downstream joins, targeting, and territory tagging.

**How it works**

**Step 1 - Add ZIP match flag**

Pulls required columns from hcp_info and sets a flag:

| **Condition** | **zip_match_flag** |
| --- | --- |
| kom_hcp_zip = hcp_zip | 1   |
| Otherwise | 0   |

This identifies addresses where VOD and KOM ZIP **match**.

CASE WHEN kom_hcp_zip = hcp_zip THEN 1 ELSE 0 END

**Step 2 - Rank addresses per HCP**

For each HCP:

- Rank **zip match first** (flag 1 > 0)
- Then rank by **latest modified date**

ROW_NUMBER() OVER (

PARTITION BY hcp_npi

ORDER BY zip_match_flag DESC, hcp_address_modified_date DESC

)

Result: rn = 1 = **best address for that HCP**

**Step 3 - Keep only the best row**

Final selection keeps:

- HCP ID + name
- Selected ZIP + state
- Whether it matched KOM (zip_match_flag)
- Corresponding hcp_vid

Returns **one record per NPI**.

**Output fields**

| **Column** | **Meaning** |
| --- | --- |
| hcp_vid | Veeva HCP ID |
| hcp_npi | NPI |
| hcp_name | Provider name |
| hcp_zip | Selected HCP ZIP |
| hcp_state | State |
| zip_match_flag | 1 = selected record matches KOM ZIP |

In [0]:
-- Create table with deduplicated HCP records
CREATE OR REPLACE TEMPORARY VIEW hcp_info_dedup AS
WITH hcp_with_zip_flag AS (
    SELECT 
        hcp_npi,
        hcp_name,
        hcp_zip,
        hcp_state,
        hcp_address_modified_date,
        kom_hcp_zip,
        hcp_vid,
        CASE 
            WHEN kom_hcp_zip = hcp_zip THEN 1 
            ELSE 0 
        END AS zip_match_flag
    FROM hcp_info
),
ranked_records AS (
    SELECT 
        *,
        -- Rank by flag first (1 before 0), then by most recent date
        ROW_NUMBER() OVER (
            PARTITION BY hcp_npi 
            ORDER BY zip_match_flag DESC, hcp_address_modified_date DESC
        ) AS rn
    FROM hcp_with_zip_flag
)
SELECT 
    hcp_vid,
    hcp_npi,
    hcp_name,
    hcp_zip,
    hcp_state,
    zip_match_flag
FROM ranked_records
WHERE rn = 1
ORDER BY hcp_npi;

In [0]:

-- One-to-one HCP -> postal code (prefers KOM ZIP match, then most recent)
SELECT
    t.hcp_npi,
    t.hcp_vid,
    t.hcp_zip
FROM (
    SELECT
        a.hcp_npi,
        b.vid__v AS hcp_vid,
        c.postal_code_cda__v AS hcp_zip,
        c.modified_date__v AS hcp_address_modified_date,
        e.PROVIDER_ZIP AS kom_hcp_zip,

        -- Ranking: prefer KOM ZIP match; then most recent address
        ROW_NUMBER() OVER (
            PARTITION BY a.hcp_npi
            ORDER BY
                CASE WHEN e.PROVIDER_ZIP = c.postal_code_cda__v THEN 1 ELSE 0 END DESC,
                c.modified_date__v DESC
        ) AS rn
    FROM com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping a
    LEFT JOIN com_edp_prd.com_raw.vod_hcp b
      ON a.hcp_npi = TRY_CAST(b.npi_num__v AS BIGINT)
    LEFT JOIN com_edp_prd.com_raw.vod_address c
      ON b.vid__v = c.entity_vid__v
     AND c.entity_type__v = 'HCP'
     AND c.record_state__v = 'VALID'
     AND c.address_status__v IN ('A','DS')
     AND c.address_verification_status__v NOT IN ('NS','U')
    LEFT JOIN com_edp_prd.com_raw.kom_providers e
      ON a.hcp_npi = e.NPI
     AND e.PROVIDER_TYPE = 'INDIVIDUAL'
) t
WHERE t.rn = 1
ORDER BY t.hcp_npi;


# HCP-HCO Affiliation View

## What This Code Does

This SQL creates a view that maps **Healthcare Professionals (HCPs)** to their affiliated **Healthcare Organizations (HCOs)** using Veeva CRM data.

---

### Key Logic

- **Joins multiple tables** to connect:
  - **HCP information:** name, NPI, location
  - **HCO information:** organization name, NPI, address
  - **Relationship data** between HCPs and HCOs

- **Filters for active affiliations only:**
  - `PARENT_HCO_STATUS__V = 'A'` &rarr; Active relationships
  - `RELATIONSHIP_TYPE__V = '7356'` &rarr; Affiliation type (excludes other relationship types)

- **Handles duplicate affiliations:**
  - When an HCP has multiple HCO affiliations, keeps only the **most recent one**
  - Prioritizes by:
    1. **Latest `modified_date__v`**
    2. **Latest `status_update_time__v`** (as tiebreaker)

- **Uses `ROW_NUMBER()` window function** to rank and select the top record per HCP

---

### Output

- **One row per HCP** with their current primary HCO affiliation
- Includes both **HCP details** (NPI, name, location) and **affiliated HCO details** (name, NPI, address)
- Provides **timestamps** for data currency tracking
---

In [0]:
-- HCP-HCO Affiliation with deduplication
CREATE OR REPLACE TEMPORARY VIEW HCP_HCO_INFO AS 
WITH RANKED_AFFILIATIONS AS (
  SELECT 
    A.hcp_npi,
    A.hcp_name,
    A.hcp_zip,
    A.hcp_state,
    c.corporate_name__v as hco_name, 
    c.npi_num__v as hco_npi,
    d.address_line_1__v as hco_address,
    d.city_cda__v as hco_city,
    d.postal_code_cda__v AS hco_zip,
    d.state_cda__v as hco_state,
    B.modified_date__v AS modified_date,
    B.status_update_time__v AS status_update_time,
    ROW_NUMBER() OVER (
      PARTITION BY A.hcp_npi 
      ORDER BY B.modified_date__v DESC NULLS LAST, 
               B.status_update_time__v DESC NULLS LAST
    ) AS rn
  FROM HCP_INFO_DEDUP AS A
  LEFT JOIN COM_EDP_PRD.COM_RAW.VOD_PARENTHCO AS B  
    ON A.HCP_VID = B.ENTITY_VID__V AND B.HIERARCHY_TYPE__V = 'HCP_HCO'
  LEFT JOIN COM_EDP_PRD.COM_RAW.VOD_HCO AS C
    ON B.PARENT_HCO_VID__V = C.VID__V
  LEFT JOIN COM_EDP_PRD.COM_RAW.VOD_ADDRESS AS D
    ON B.PARENT_HCO_VID__V = D.ENTITY_VID__V AND D.ENTITY_TYPE__V = 'HCO' 
  LEFT JOIN COM_EDP_PRD.CMPA_INSIGHTS_INTERNAL_SCHEMA.ZIP_TO_TERRITORY_MAPPING AS E
    ON D.POSTAL_CODE_CDA__V = E.ZIPCODE
  LEFT JOIN COM_EDP_PRD.COM_RAW.KOM_PROVIDERS AS F
    ON C.NPI_NUM__V = F.NPI AND F.PROVIDER_TYPE = 'ORGANIZATION'
  LEFT JOIN COM_EDP_PRD.CMPA_INSIGHTS_INTERNAL_SCHEMA.ZIP_TO_TERRITORY_MAPPING AS G
    ON F.PROVIDER_ZIP = G.ZIPCODE
  WHERE 
    B.PARENT_HCO_STATUS__V = 'A' -- PARENT_HCO_STATUS = ACTIVE
    AND B.RELATIONSHIP_TYPE__V = '7356' -- RELATIONSHIP_TYPE = AFFILIATION
)
SELECT 
  hcp_npi,
  hcp_name,
  hcp_zip,
  hcp_state,
  hco_name,
  hco_npi,
  hco_address,
  hco_city,
  hco_zip,
  hco_state
FROM RANKED_AFFILIATIONS
WHERE rn = 1;

In [0]:
select *
from hcp_hco_info

In [0]:
-- HCP-HCO Affiliation with deduplication
CREATE OR REPLACE TEMPORARY VIEW HCP_HCO_INFO AS 
WITH RANKED_AFFILIATIONS AS (
  SELECT 
    A.hcp_npi,
    c.vid__v as hco_vid,
    c.corporate_name__v as hco_name, 
    c.npi_num__v as hco_npi,
    B.modified_date__v AS modified_date,
    B.status_update_time__v AS status_update_time,
    ROW_NUMBER() OVER (
      PARTITION BY A.hcp_npi 
      ORDER BY B.modified_date__v DESC NULLS LAST, 
               B.status_update_time__v DESC NULLS LAST
    ) AS rn
  FROM HCP_INFO_DEDUP AS A
  LEFT JOIN COM_EDP_PRD.COM_RAW.VOD_PARENTHCO AS B
    ON A.HCP_VID = B.ENTITY_VID__V AND B.HIERARCHY_TYPE__V = 'HCP_HCO'
  LEFT JOIN COM_EDP_PRD.COM_RAW.VOD_HCO AS C
    ON B.PARENT_HCO_VID__V = C.VID__V
  WHERE 
    B.PARENT_HCO_STATUS__V = 'A' -- PARENT_HCO_STATUS = ACTIVE
    AND B.RELATIONSHIP_TYPE__V = '7356' -- RELATIONSHIP_TYPE = AFFILIATION
)
SELECT 
  hcp_npi,
  hco_npi,
  hco_vid,
  hco_name
FROM RANKED_AFFILIATIONS
WHERE rn = 1;

In [0]:
-- create or replace temporary view HCP_HCO_WITH_HCO_ZIP AS 
with hcp_hco as (
  select * from hcp_hco_info
),
hco_zip_v1 as (
  select distinct
    a.vid__v as hco_vid,
    b.postal_code_cda__v as hco_postal_code,
    b.modified_date__v,
    row_number() over (partition by a.vid__v order by b.modified_date__v desc) as rn
from com_raw.vod_hco a
join com_raw.vod_address b
    on b.entity_vid__v = a.vid__v
   and b.entity_type__v = 'HCO'
   AND b.record_state__v = 'VALID'
   AND b.address_status__v IN ('A','DS')
   AND b.address_verification_status__v NOT IN ('NS', 'U')
where a.vid__v in (select distinct hco_vid from hcp_hco where hco_vid is not null)
),
hco_zip_v2 as (
  select distinct hco_vid, hco_postal_code from hco_zip_v1 where rn = 1
)
select a.*, b.hco_postal_code
from hcp_hco as a
left join hco_zip_v2 as b on a.hco_vid = b.hco_vid 

In [0]:

-- One-to-one HCP -> postal code (prefers KOM ZIP match, then most recent)
SELECT
    t.hcp_npi,
    t.hcp_vid,
    t.hcp_zip
FROM (
    SELECT
        a.hcp_npi,
        b.vid__v AS hcp_vid,
        c.postal_code_cda__v AS hcp_zip,
        c.modified_date__v AS hcp_address_modified_date,
        e.PROVIDER_ZIP AS kom_hcp_zip,

        -- Ranking: prefer KOM ZIP match; then most recent address
        ROW_NUMBER() OVER (
            PARTITION BY a.hcp_npi
            ORDER BY
                CASE WHEN e.PROVIDER_ZIP = c.postal_code_cda__v THEN 1 ELSE 0 END DESC,
                c.modified_date__v DESC
        ) AS rn
    FROM com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping a
    LEFT JOIN com_edp_prd.com_raw.vod_hcp b
      ON a.hcp_npi = TRY_CAST(b.npi_num__v AS BIGINT)
    LEFT JOIN com_edp_prd.com_raw.vod_address c
      ON b.vid__v = c.entity_vid__v
     AND c.entity_type__v = 'HCP'
     AND c.record_state__v = 'VALID'
     AND c.address_status__v IN ('A','DS')
     AND c.address_verification_status__v NOT IN ('NS','U')
) t
WHERE t.rn = 1
ORDER BY t.hcp_npi;


In [0]:
%sql
-- Query to find HCP and HCO mapping for NPI 1609305853
-- This follows the same logic as your view but for a specific NPI not in target list

WITH hcp_base AS (
  -- Get HCP basic info from Veeva
  SELECT
      1609305853 AS hcp_npi,
      b.vid__v AS hcp_vid,
      b.first_name__v AS hcp_first_name,
      b.last_name__v AS hcp_last_name,
      CONCAT(b.first_name__v, ' ', b.last_name__v) AS hcp_name
  FROM com_edp_prd.com_raw.vod_hcp b
  WHERE TRY_CAST(b.npi_num__v AS BIGINT) = 1609305853
),

hcp_addresses AS (
  -- Get all addresses for this HCP
  SELECT
      h.*,
      c.modified_date__v AS hcp_address_modified_date,
      c.address_line_1__v AS hcp_address_line_1,
      c.postal_code_cda__v AS hcp_zip,
      c.city_cda__v AS hcp_city,
      c.state_cda__v AS hcp_state,
      d.territory_name AS hcp_terr,
      d.region_name AS hcp_region,
      c.address_type__v AS hcp_address_type,
      c.address_verification_status__v AS hcp_address_verification_status,
      c.vid__v AS hcp_address_vid,
      c.record_state__v,
      c.address_status__v,
      CASE
        WHEN c.record_state__v = 'VALID' 
         AND c.address_status__v IN ('A','DS')
         AND c.address_verification_status__v NOT IN ('NS', 'U')
        THEN 1
        ELSE 0
      END AS hcp_address_flag
  FROM hcp_base h
  LEFT JOIN com_edp_prd.com_raw.vod_address c
    ON h.hcp_vid = c.entity_vid__v
   AND c.entity_type__v = 'HCP'
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping d
    ON c.postal_code_cda__v = d.zipcode
),

kom_addresses AS (
  -- Get KOM provider addresses
  SELECT
      e.NPI,
      e.PROVIDER_ADDRESS AS kom_hcp_address_line_1,
      e.PROVIDER_ZIP AS kom_hcp_zip,
      e.PROVIDER_CITY AS kom_hcp_city,
      e.PROVIDER_STATE AS kom_hcp_state,
      f.territory_name AS kom_hcp_terr,
      f.region_name AS kom_hcp_region
  FROM com_edp_prd.com_raw.kom_providers e
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping f
    ON e.PROVIDER_ZIP = f.zipcode
  WHERE e.NPI = 1609305853
    AND e.PROVIDER_TYPE = 'INDIVIDUAL'
),

combined_hcp AS (
  -- Combine Veeva and KOM data
  SELECT
      ha.*,
      ka.kom_hcp_address_line_1,
      ka.kom_hcp_zip,
      ka.kom_hcp_city,
      ka.kom_hcp_state,
      ka.kom_hcp_terr,
      ka.kom_hcp_region,
      CASE 
        WHEN ka.kom_hcp_zip = ha.hcp_zip THEN 1 
        ELSE 0 
      END AS zip_match_flag
  FROM hcp_addresses ha
  LEFT JOIN kom_addresses ka ON ha.hcp_npi = ka.NPI
),

hcp_hco_affiliations AS (
  -- Get HCO affiliations for this HCP
  SELECT 
      ch.hcp_npi,
      ch.hcp_name,
      ch.hcp_zip,
      ch.hcp_state,
      ch.hcp_address_flag,
      ch.zip_match_flag,
      c.corporate_name__v as hco_name, 
      c.npi_num__v as hco_npi,
      d.address_line_1__v as hco_address,
      d.city_cda__v as hco_city,
      d.postal_code_cda__v AS hco_zip,
      d.state_cda__v as hco_state,
      B.modified_date__v AS affiliation_modified_date,
      B.status_update_time__v AS affiliation_status_update_time,
      B.parent_hco_status__v,
      B.relationship_type__v,
      -- Rank to get most recent affiliation
      ROW_NUMBER() OVER (
        PARTITION BY ch.hcp_npi 
        ORDER BY 
          ch.zip_match_flag DESC,  -- Prefer addresses where Veeva and KOM match
          ch.hcp_address_flag DESC, -- Then prefer valid addresses
          ch.hcp_address_modified_date DESC, -- Then most recent address
          B.modified_date__v DESC NULLS LAST, 
          B.status_update_time__v DESC NULLS LAST
      ) AS rn
  FROM combined_hcp ch
  LEFT JOIN COM_EDP_PRD.COM_RAW.VOD_PARENTHCO AS B
      ON ch.HCP_VID = B.ENTITY_VID__V 
     AND B.HIERARCHY_TYPE__V = 'HCP_HCO'
     AND B.PARENT_HCO_STATUS__V = 'A' -- Active affiliations only
     AND B.RELATIONSHIP_TYPE__V = '7356' -- Affiliation type
  LEFT JOIN COM_EDP_PRD.COM_RAW.VOD_HCO AS C
      ON B.PARENT_HCO_VID__V = C.VID__V
  LEFT JOIN COM_EDP_PRD.COM_RAW.VOD_ADDRESS AS D
      ON B.PARENT_HCO_VID__V = D.ENTITY_VID__V 
     AND D.ENTITY_TYPE__V = 'HCO'
)

-- Final output
SELECT 
    'All Addresses for HCP' AS query_type,
    hcp_npi,
    hcp_name,
    hcp_address_line_1,
    hcp_city,
    hcp_state,
    hcp_zip,
    hcp_terr,
    record_state__v,
    address_status__v,
    hcp_address_flag,
    hcp_address_modified_date
FROM hcp_addresses
ORDER BY hcp_address_flag DESC, hcp_address_modified_date DESC;

### Primary Affiliation (ProcDNA's Logic)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tl_hcps AS
select distinct 
a.hcp_npi, 
b.vid__v as hcp_vid, 
concat(a.hcp_first_name, ' ', a.hcp_last_name) as hcp_name,
COALESCE(sg.name, b.primary_specialty_group__v) AS hcp_primary_specialty_group__v
from com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping a
left join com_edp_prd.com_raw.vod_hcp b 
  on a.hcp_npi = try_cast(b.npi_num__v as bigint)
LEFT JOIN com_edp_prd.com_raw.vod_references sg
    ON b.primary_specialty_group__v = sg.code
    AND sg.reference_type = 'SpecialtyGroup'

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW hcp_hco_mapping AS
select a.*, b.parent_hco_vid__v as hco_vid, b.modified_date__v as parent_hco_modified_date, b.status_update_time__v as parent_hco_status_update_time, c.corporate_name__v as hco_name, c.npi_num__v as hco_npi, coalesce(d.name, c.specialty_1__v) as hco_specialty, 
case when a.hcp_primary_specialty_group__v = coalesce(d.name, c.specialty_1__v) then 1 else 0 end as specialty_same_flag
from tl_hcps as a
join com_edp_prd.com_raw.vod_parenthco as b
on a.hcp_vid = b.entity_vid__v
left join com_edp_prd.com_raw.vod_hco as c
on b.parent_hco_vid__v = c.vid__v
left join com_edp_prd.com_raw.vod_references as d
on c.specialty_1__v = d.code and d.reference_type = 'Specialty'
WHERE b.hierarchy_type__v IN ('HCP_HCO')
AND b.parent_hco_status__v = 'A' -- Active
AND b.relationship_type__v = '7356'; -- Affiliation

In [0]:
select * from hcp_hco_mapping

In [0]:
%python
# Databricks / PySpark -> pandas implementation
import pandas as pd
import numpy as np

# 1) Read Spark temp view into a Spark DataFrame, then convert to pandas as requested
#    (this will collect the view into driver memory — be mindful of size)
spark_df = spark.table("hcp_hco_mapping")
pdf = spark_df.toPandas()

# -------------------------
# Preprocessing
# -------------------------
# Normalize column names if necessary (optional)
# pdf.columns = [c.strip() for c in pdf.columns]

# Ensure the key columns exist
required_cols = [
    "hcp_vid", "hco_vid", "specialty_same_flag",
    "parent_hco_modified_date"
]
missing = [c for c in required_cols if c not in pdf.columns]
if missing:
    raise ValueError(f"Missing required columns in temp view: {missing}")

# Convert specialty flag to numeric (0/1) safely
pdf["specialty_same_flag"] = pd.to_numeric(pdf["specialty_same_flag"], errors="coerce").fillna(0).astype(int)

# Parse the parent_hco_modified_date to datetime (coerce errors -> NaT)
pdf["parent_hco_modified_date"] = pd.to_datetime(pdf["parent_hco_modified_date"], errors="coerce")

# Optional: if your dates are strings with timezone or weird formats, adjust parsing above.

# -------------------------
# Helper function to determine primary(s) for a single HCP group
# -------------------------
def pick_primary_for_group(gdf: pd.DataFrame) -> pd.DataFrame:
    """
    Given rows for one hcp_vid, return same rows with columns:
      - is_primary_hco (bool)
      - primary_reason (str)
    """
    # initialize
    gdf = gdf.copy()
    gdf["is_primary_hco"] = False
    gdf["primary_reason"] = None

    # if only one hco mapped -> mark it primary
    if len(gdf) == 1:
        gdf.loc[gdf.index, "is_primary_hco"] = True
        gdf.loc[gdf.index, "primary_reason"] = "only_hco"
        return gdf

    # find rows where specialty_same_flag == 1
    flag_rows = gdf[gdf["specialty_same_flag"] == 1]

    if len(flag_rows) == 1:
        # exactly one with the flag -> primary
        idx = flag_rows.index[0]
        gdf.at[idx, "is_primary_hco"] = True
        gdf.at[idx, "primary_reason"] = "specialty_same_flag_unique"
        return gdf

    # else: either zero with flag==1 or more than one -> fallback to most recent parent_hco_modified_date
    # find max date (NaT-safe)
    # If all dates are NaT, max_date will be NaT; in that case, treat as tie -> mark all as primary per your spec
    max_date = gdf["parent_hco_modified_date"].max()

    if pd.isna(max_date):
        # can't decide by date, keep all as primary (user said "keeping both of them as the primary")
        gdf["is_primary_hco"] = True
        gdf["primary_reason"] = "tie_all_dates_null_or_undetermined"
        return gdf

    # select rows whose parent_hco_modified_date equals the max_date (use >= comparison with tolerance if needed)
    # If you worry about microsecond differences, you can normalize to date or round.
    selected = gdf[gdf["parent_hco_modified_date"] == max_date]
    gdf.loc[selected.index, "is_primary_hco"] = True
    # provide reason depending on whether flag rows existed
    if len(flag_rows) > 1:
        gdf.loc[selected.index, "primary_reason"] = "multiple_flag_rows_fallback_most_recent_date"
    elif len(flag_rows) == 0:
        gdf.loc[selected.index, "primary_reason"] = "no_flag_rows_fallback_most_recent_date"
    else:
        # should not reach here logically
        gdf.loc[selected.index, "primary_reason"] = "fallback_most_recent_date"

    return gdf

# -------------------------
# Apply per-HCP grouping
# -------------------------
result_parts = []
grouped = pdf.groupby("hcp_vid", dropna=False)  # include NaN hcp_vid rows if any
for hcp_vid, g in grouped:
    res = pick_primary_for_group(g)
    result_parts.append(res)

result_pdf = pd.concat(result_parts, axis=0)

# Optional: re-order columns to put is_primary_hco and primary_reason near the end or wherever you prefer
cols = list(pdf.columns)  # original order
# remove if already present to avoid duplicates
cols = [c for c in cols if c not in ("is_primary_hco", "primary_reason")]
cols += ["is_primary_hco", "primary_reason"]
result_pdf = result_pdf[cols]

# Reset index (optional) and show
result_pdf = result_pdf.reset_index(drop=True)


In [0]:
%python
result_pdf[result_pdf['is_primary_hco'] == True]['hcp_vid'].nunique()


In [0]:
%python
result_pdf_primary_hco = result_pdf[result_pdf['is_primary_hco'] == True].copy()

In [0]:
%python
result_pdf_primary_hco.shape

In [0]:
%python
display(result_pdf_primary_hco.head())

### If still more than one affiliation is there order by hcp id and keep the first one

In [0]:
%python
result_pdf_primary_hco['hcp_npi'].nunique()

In [0]:
%python
# Ensure hco_vid is sortable (convert to string if it’s not numeric)
result_pdf_primary_hco['hco_vid'] = result_pdf_primary_hco['hco_vid'].astype(str)

# Sort first by hcp_vid and then by hco_vid ascending
result_pdf_primary_hco = result_pdf_primary_hco.sort_values(by=['hcp_vid', 'hco_vid'], ascending=[True, True])

# Drop duplicates — keep only the first HCO per HCP
result_pdf_primary_hco_unique = result_pdf_primary_hco.drop_duplicates(subset=['hcp_vid'], keep='first')

# Reset index for cleanliness
result_pdf_primary_hco_unique = result_pdf_primary_hco_unique.reset_index(drop=True)

# Optional: sanity check
# print("Before:", result_pdf_primary_hco['hcp_vid'].nunique(), "unique HCPs with", len(result_pdf_primary_hco), "rows")
# print("After :", result_pdf_primary_hco_unique['hcp_vid'].nunique(), "unique HCPs with", len(result_pdf_primary_hco_unique), "rows")

# Display sample to confirm
# result_pdf_primary_hco_unique[['hcp_vid', 'hco_vid', 'is_primary_hco', 'primary_reason']].head(20)

In [0]:
%python
display(result_pdf_primary_hco_unique)

In [0]:
%python
result_pdf_primary_hco_unique['hcp_npi'].nunique()

In [0]:
%python
result_pdf_primary_hco_unique['procdna_flag'] = 1

In [0]:
%python
spark_df = spark.createDataFrame(result_pdf_primary_hco_unique[['hcp_npi', 'hco_npi', 'procdna_flag']])
spark_df.createOrReplaceTempView("hcp_hco_primary_unique_view")


### Summary

In [0]:
create or replace temporary view hcp_hco_concised as 
with t1 as (
  select 
  -- distinct hcp_npi, hcp_zip, hcp_state, kom_hcp_zip, kom_hcp_state, hco_npi, hco_zip, hco_state, kom_hco_zip, kom_hco_state
  distinct *
from hcp_hco_info
)
select a.*, b.procdna_flag
from t1 as a
left join hcp_hco_primary_unique_view as b on a.hcp_npi = b.hcp_npi and a.hco_npi = b.hco_npi

In [0]:
select * from hcp_hco_concised limit 5

In [0]:
select hcp_npi, count(distinct hco_vid)
from hcp_hco_concised
where hco_validity_flag = 1
group by 1
order by 2 desc

In [0]:
WITH hcp_hco_counts AS (
    SELECT 
        hcp_npi,
        COUNT(DISTINCT hco_vid) AS distinct_hco_count
    FROM hcp_hco_concised
    WHERE hco_validity_flag = 1
    GROUP BY hcp_npi
)

SELECT
    CASE
        WHEN distinct_hco_count = 1 THEN '1 HCO (Single Affiliation)'
        WHEN distinct_hco_count = 2 THEN '2 HCOs'
        WHEN distinct_hco_count = 3 THEN '3 HCOs'
        WHEN distinct_hco_count = 4 THEN '4 HCOs'
        WHEN distinct_hco_count = 5 THEN '5 HCOs'
        WHEN distinct_hco_count BETWEEN 6 AND 8 THEN '6-8 HCOs'
        WHEN distinct_hco_count BETWEEN 9 AND 10 THEN '9-10 HCOs'
        WHEN distinct_hco_count BETWEEN 11 AND 15 THEN '10-15 HCOs'
        WHEN distinct_hco_count > 15 THEN '15+ HCOs'
    END AS bucket,
    COUNT(*) AS distinct_hcp_count
FROM hcp_hco_counts
GROUP BY 1
ORDER BY 
    CASE 
        WHEN bucket = '1 HCO (Single Affiliation)' THEN 1
        WHEN bucket = '2 HCOs' THEN 2
        WHEN bucket = '3 HCOs' THEN 3
        WHEN bucket = '4 HCOs' THEN 4
        WHEN bucket = '5 HCOs' THEN 5
        WHEN bucket = '6-8 HCOs' THEN 6
        WHEN bucket = '9-10 HCOs' THEN 7
        WHEN bucket = '10-15 HCOs' THEN 8
        WHEN bucket = '15+ HCOs' THEN 9
    END;


In [0]:
select hcp_npi, count(distinct hco_terr)
from hcp_hco_concised
where hco_validity_flag = 1
group by 1
order by 2 desc

In [0]:
WITH per_hcp AS (
  SELECT
      hcp_npi,
      COUNT(DISTINCT hco_terr) AS terr_cnt
  FROM hcp_hco_concised
  WHERE hco_validity_flag = 1
  GROUP BY hcp_npi
)
SELECT
  CASE terr_cnt
    WHEN 1 THEN '1 Territory'
    WHEN 2 THEN '2 Territories'
    WHEN 3 THEN '3 Territories'
    WHEN 4 THEN '4 Territories'
    WHEN 5 THEN '5 Territories'
    WHEN 6 THEN '6 Territories'
  END AS bucket,
  COUNT(*) AS distinct_hcp_count
FROM per_hcp
GROUP BY terr_cnt
ORDER BY terr_cnt;


In [0]:
with t1 as (
  select *,
  case when hcp_zip = kom_hcp_zip then 1 else 0 end as kom_hcp_zip_qc
  from hcp_hco_concised
),
t2 as (
  select *, 
case when hcp_zip = hco_zip then 1 else 0 end as hcp_hco_zip_check
from t1
where kom_hcp_zip_qc = 1
),
hcp_hco_counts AS (
    SELECT 
        hcp_npi,
        COUNT(DISTINCT hco_vid) AS distinct_hco_count
    FROM t2
    WHERE hco_validity_flag = 1 and kom_hcp_zip_qc = 1 and hcp_hco_zip_check = 1
    GROUP BY hcp_npi
)

SELECT
    CASE
        WHEN distinct_hco_count = 1 THEN '1 HCO (Single Affiliation)'
        WHEN distinct_hco_count = 2 THEN '2 HCOs'
        WHEN distinct_hco_count = 3 THEN '3 HCOs'
        WHEN distinct_hco_count = 4 THEN '4 HCOs'
        WHEN distinct_hco_count = 5 THEN '5 HCOs'
        WHEN distinct_hco_count BETWEEN 6 AND 8 THEN '6-8 HCOs'
        WHEN distinct_hco_count BETWEEN 9 AND 10 THEN '9-10 HCOs'
        WHEN distinct_hco_count BETWEEN 11 AND 15 THEN '10-15 HCOs'
        WHEN distinct_hco_count > 15 THEN '15+ HCOs'
    END AS bucket,
    COUNT(*) AS distinct_hcp_count
FROM hcp_hco_counts
GROUP BY 1
ORDER BY 
    CASE 
        WHEN bucket = '1 HCO (Single Affiliation)' THEN 1
        WHEN bucket = '2 HCOs' THEN 2
        WHEN bucket = '3 HCOs' THEN 3
        WHEN bucket = '4 HCOs' THEN 4
        WHEN bucket = '5 HCOs' THEN 5
        WHEN bucket = '6-8 HCOs' THEN 6
        WHEN bucket = '9-10 HCOs' THEN 7
        WHEN bucket = '10-15 HCOs' THEN 8
        WHEN bucket = '15+ HCOs' THEN 9
    END;

In [0]:
with t1 as (
  select *,
  case when hcp_zip = kom_hcp_zip then 1 else 0 end as kom_hcp_zip_qc
  from hcp_hco_concised
),
t2 as (
  select *, 
case when hcp_zip = hco_zip then 1 else 0 end as hcp_hco_zip_check
from t1
where kom_hcp_zip_qc = 1 
),
per_hcp AS (
  SELECT
      hcp_npi,
      COUNT(DISTINCT hco_terr) AS terr_cnt
  FROM t2
  WHERE hco_validity_flag = 1 and kom_hcp_zip_qc = 1 and hcp_hco_zip_check = 1
  GROUP BY hcp_npi
)
SELECT
  CASE terr_cnt
    WHEN 1 THEN '1 Territory'
    WHEN 2 THEN '2 Territories'
    WHEN 3 THEN '3 Territories'
    WHEN 4 THEN '4 Territories'
    WHEN 5 THEN '5 Territories'
    WHEN 6 THEN '6 Territories'
  END AS bucket,
  COUNT(*) AS distinct_hcp_count
FROM per_hcp
GROUP BY terr_cnt
ORDER BY terr_cnt;

In [0]:
select * from hcp_hco_concised

In [0]:
with t1 as (
  select distinct hcp_npi,
  case when hcp_zip = kom_hcp_zip then 1 else 0 end as kom_vod_hcp_zip_qc
  from hcp_hco_concised
),
t2 as (
  select distinct hcp_npi, kom_vod_hcp_zip_qc
from t1
where kom_vod_hcp_zip_qc = 1
)
select distinct a.hcp_npi, a.hcp_vid, a.hcp_name, a.hcp_zip, a.hco_city, a.hcp_state, a.hco_vid, a.hco_npi, a.hco_name, a.hco_city, a.hco_zip, a.hco_state, a.hcp_address_flag, a.hco_validity_flag,
b.kom_vod_hcp_zip_qc
from hcp_hco_concised as a
left join t2 as b on a.hcp_npi = b.hcp_npi

In [0]:
with t1 as (
  select distinct hcp_npi,
  case when hcp_zip = kom_hcp_zip then 1 else 0 end as kom_vod_hcp_zip_qc
  from hcp_hco_concised
),
t2 as (
  select distinct hcp_npi, kom_vod_hcp_zip_qc
from t1
where kom_vod_hcp_zip_qc = 1
),
t3 as (
  select distinct a.hcp_npi, a.hcp_vid, a.hcp_name, a.hcp_zip, a.hco_city, a.hcp_state, a.hco_vid, a.hco_npi, a.hco_name, a.hco_city, a.hco_zip, a.hco_state, a.hcp_address_flag, a.hco_validity_flag,
b.kom_vod_hcp_zip_qc
from hcp_hco_concised as a
left join t2 as b on a.hcp_npi = b.hcp_npi
)
select count(distinct hcp_npi) from t3 where hcp_vid is not null and hcp_address_flag = 0

In [0]:
select count(distinct hcp_npi)
from t1
where hcp_vid is not null and hcp_address_flag = 0